# HW-03: Classification with Logistic Regression

**Important:** You will implement all functions in `ds.py` and then *use them here*.

Workflow:
1. Implement functions in `ds.py`
2. Restart kernel (or reload the module)
3. Run the cells below to execute your end-to-end classification pipeline

Do **not** copy/paste solutions into this notebook. Use `import ds` and call your functions.


## 0. Setup

Run this cell first. If you edit `ds.py`, rerun this cell to reload it.

In [ ]:
from __future__ import annotations

from pathlib import Path

import importlib
import numpy as np
import pandas as pd

import ds

# If you edited ds.py, rerun this cell (or restart kernel) to pick up changes
importlib.reload(ds)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)


## 1. Load the data

This homework is designed for a COVID dataset with a binary outcome.

- Dataset filename: `covid-19-symptoms-dataset.csv`
- Target column: `covid_positive` (0/1)


In [ ]:
df = ds.load_csv("data/covid-19-symptoms-dataset.csv")
df.head()

## 2. Basic cleaning (missing values)

Use your `handle_missing_values()` implementation.

In [ ]:
df_clean = ds.handle_missing_values(df)
df_clean.isna().sum()


## 3. Feature engineering: create a categorical column

Create `age_group` from `age` **before** splitting.

Suggested bins (edit if you want):
- 0–17
- 18–29
- 30–44
- 45–59
- 60+


In [ ]:
df_fe = ds.create_categorical_column(
    df_clean,
    source_col='age',
    new_col='age_group',
    bins=[#Add bins here],
    labels=[#Add labels here],
    right=False,
)

df_fe[['age', 'age_group']].head(10)


## 4. Train/test split

Use your `train_test_split_df()` implementation. The target column for this dataset is `covid_positive`.


In [ ]:
TARGET_COL = 'covid_positive'

train_df, test_df = ds.train_test_split_df(df_fe, target_col=TARGET_COL, test_size=0.2, random_state=42)
train_df.shape, test_df.shape


## 5. Preprocessing

### 5a. Standardize numeric columns (train statistics only)

For this dataset, one continuous numeric column is `age` (but why would we not want to use this column?). Note however that `fever` is also continuous. 
The symptom columns are already 0/1 and do not need standardization.


In [ ]:
NUMERIC_COLS = ['age']
train_std, test_std = ds.standardize_numeric_columns(train_df, test_df, columns=NUMERIC_COLS)

train_std[NUMERIC_COLS].describe().T


### 5b. One-hot encode categorical columns

Encode `age_group` (and any others you choose).

In [ ]:
CATEGORICAL_COLS = ['age_group']
train_enc, test_enc = ds.encode_categorical_columns(train_std, test_std, columns=CATEGORICAL_COLS, drop_first=False)

train_enc.head()


## 6. Select features and target

Decide which columns are features. A common default is: **everything except the target**.

If you want to drop columns, do it by editing `FEATURE_COLS` below.

In [ ]:
FEATURE_COLS = [c for c in train_enc.columns if c != TARGET_COL]

X_train, y_train = ds.select_features_and_target(train_enc, FEATURE_COLS, TARGET_COL)
X_test, y_test = ds.select_features_and_target(test_enc, FEATURE_COLS, TARGET_COL)

X_train.shape, y_train.shape, X_test.shape, y_test.shape


## 7. Train logistic regression

Train the model using your `train_logistic_regression()` function.
You may adjust `C` and `max_iter` if needed.

In [ ]:
model = ds.train_logistic_regression(X_train, y_train, C=1.0, max_iter=1000)
model


## 8. Predict probabilities and labels

Use `predict_probabilities()` then `predict_labels()`.
Try changing the threshold to see how precision/recall trade off.

In [ ]:
proba_test = ds.predict_probabilities(model, X_test)
y_pred = ds.predict_labels(proba_test, threshold=0.5)

proba_test[:10], y_pred[:10]


## 9. Evaluate

Compute metrics and confusion matrix counts using your functions.

In [ ]:
metrics = ds.classification_metrics(y_test, y_pred)
cm = ds.confusion_matrix_counts(y_test, y_pred)

metrics, cm


## 10. Reflection prompts (write answers in Markdown)

- Which metric matters most for a COVID outcome and why?
- If false negatives are more costly than false positives, what might you change?
- How does changing the threshold affect precision and recall?
